In [79]:
import os
from loader import read_sentences
from candidates import BKTree, SymSpell
from ngram_model import KNgramModel
from corrections import SpellCorrector
from spylls.hunspell import Dictionary
import random
from tqdm import tqdm
from typing import Optional
from wordfreq import zipf_frequency
import numpy as np


In [80]:
max_distance = 1

language = "es"
misfit_file = True

data_dir = "./data"

hunspell_dir = os.path.join(data_dir, "hunspell")
hunspell_dict_dir = os.path.join(hunspell_dir, "es_ES")
dict_path = os.path.join(hunspell_dir, "es_ES_unmunched_words.txt")

sentences_path = os.path.join(data_dir, f"{language}_sentences.txt")
test_sentences_path = os.path.join(data_dir, f"{language}_test_sentences.txt")

model_dir = os.path.join("./models", f"distance_{max_distance}")

bk_tree_path = os.path.join(model_dir, "bk_tree.pkl")
sym_spell_path = os.path.join(model_dir, "sym_spell.pkl")

forward_lm_path = os.path.join(model_dir, "forward_lm.pkl")
backward_lm_path = os.path.join(model_dir, "backward_lm.pkl")


In [81]:
lexicon = Dictionary.from_files(hunspell_dict_dir)


In [82]:
# tree = BKTree.load(bk_tree_path)


In [83]:
# result = tree.search("coch", 2)
# tree_set = {word for word, _ in result}
# print(tree_set)


In [84]:
sym_spell = SymSpell.load(sym_spell_path)


In [85]:
result = sym_spell.search("coch", 2)
sym_set = {word for word, _ in result}
print(sym_set)


{'col', 'cocó', 'ocho', 'coy', 'cocí', 'cloc', 'cocho', 'coz', 'cochi', 'oc', 'cocha', 'coca', 'cok', 'coche', 'oh', 'ch', 'oca', 'con', 'coco', 'coa', 'cor'}


In [86]:
# print(len(tree_set ^ sym_set))


In [87]:
forward_lm = KNgramModel.load(forward_lm_path)


In [88]:
top_n = 30

for k, v in sorted(forward_lm.cont_count.items(), key=lambda x: x[1], reverse=True)[:top_n]:
	print(f"{k}: {v}")
	

</s>: 29083
de: 9872
y: 8215
a: 8135
en: 7402
que: 5570
el: 5129
por: 4825
la: 4755
no: 4450
con: 4407
para: 3803
<unk>: 3425
un: 3263
al: 3038
es: 2990
del: 2905
pero: 2600
como: 2571
una: 2496
se: 2486
lo: 2204
los: 2102
me: 2003
más: 1988
las: 1788
su: 1722
o: 1717
si: 1613
está: 1503


In [89]:
backward_lm = KNgramModel.load(backward_lm_path)


In [90]:
print(sym_spell.max_dist)


1


In [91]:
spell_corrector = SpellCorrector(
	lexicon=lexicon,
	candidate_gen=sym_spell,
	forward_lm=forward_lm,
	backward_lm=backward_lm,
	max_distance=sym_spell.max_dist
)


In [92]:
text = "Q tal estás?"

spell_corrector.correct_text(
	text=text,
	unigram_weight=1.0
)


'Y tal estás?'

In [93]:
test_sentences = read_sentences(test_sentences_path)


In [94]:
print(f"{len(test_sentences):,}")


59,474


In [95]:
class SpellCorrectionEvaluator:
	LETTERS = tuple("abcdefghijklmnopqxyzáéíóúüñ")
	MAX_ERROR_ATTEMPTS = 10

	def __init__(self, spell_corrector: SpellCorrector, distance_weights: Optional[dict[int, float]] = None, operation_weights: Optional[dict[str, float]] = None, seed: int = 42):
		self.spell_corrector = spell_corrector
		self.candidate_gen = spell_corrector.candidate_gen
		self.max_distance = spell_corrector.max_distance
		self.lexicon = spell_corrector.lexicon

		distance_weights = distance_weights or {
			1: 0.75,
			2: 0.25
		}

		self.edit_distances = []
		self.edit_distance_weights = []

		for dist, weight in distance_weights.items():
			if dist <= self.max_distance:
				self.edit_distances.append(dist)
				self.edit_distance_weights.append(weight)

		self.operation_weights = operation_weights or {
			"substitute": 0.45,
			"insert": 0.25,
			"delete": 0.15,
			"transpose": 0.15
		}

		self.rng = random.Random(seed)
		self.np_rng = np.random.default_rng(seed)

	def random_typo(self, word: str, k: int) -> str:
		chars = list(word)

		for _ in range(k):
			n = len(chars)

			available_ops = []
			weights = []
		
			if n > 0:
				available_ops.extend(["delete", "substitute"])
				weights.extend([
					self.operation_weights["delete"],
					self.operation_weights["substitute"],
				])
				
			available_ops.append("insert")
			weights.append(self.operation_weights["insert"])
			
			if n > 1:
				available_ops.append("transpose")
				weights.append(self.operation_weights["transpose"])

			op = self.rng.choices(available_ops, weights=weights, k=1)[0]

			if op == "delete" and n > 0:
				i = self.rng.randrange(n)
				del chars[i]

			elif op == "insert":
				c = self.rng.choice(self.LETTERS)
				i = self.rng.randrange(n + 1)
				chars.insert(i, c)

			elif op == "substitute":
				i = self.rng.randrange(n)
				chars[i] = self.rng.choice(self.LETTERS)

			elif op == "transpose" and n > 1:
				i = self.rng.randrange(n - 1)
				chars[i], chars[i + 1] = chars[i + 1], chars[i]

		result = "".join(chars)
		return result if result != word else self.random_typo(word, k)

	def generate_error_word(self, word: str, edit_distance: int):
		for _ in range(self.MAX_ERROR_ATTEMPTS):
			typo = self.random_typo(word, edit_distance)
			if not self.lexicon.lookup(typo):
				return typo
		return typo
	
	def sample_error_indices(self, sentence: list[str], errors_per_sentence: int) -> list[int]:
		valid_indices = []

		for i, word in enumerate(sentence):
			if word != "<num>":
				valid_indices.append(i)

		if len(valid_indices) <= errors_per_sentence:
			return valid_indices

		weights = []

		for idx in valid_indices:
			z = zipf_frequency(sentence[idx].lower(), "es")
			weights.append(max(z, 1e-3))

		weights = np.asarray(weights, dtype=float)
		weights /= weights.sum()

		chosen = self.np_rng.choice(
			len(valid_indices),
			size=errors_per_sentence,
			replace=False,
			p=weights,
		)

		return [valid_indices[i] for i in chosen]
	
	def generate_errors(self, data: list[list[str]], errors_per_sentence: int) -> tuple[list[list[str]], list[list[int]]]:
		corrupted = [sentence.copy() for sentence in data]
		error_locations = []

		for sentence in tqdm(corrupted, desc="Generating errors"):
			n_words = len(sentence)

			indices = self.sample_error_indices(
				sentence,
				min(errors_per_sentence, n_words),
			)

			error_locations.append(indices)

			for idx in indices:
				# Normaliza internamente los pesos si no suman 1
				edit_distance = self.rng.choices(self.edit_distances, weights=self.edit_distance_weights, k=1)[0]
				sentence[idx] = self.generate_error_word(
					sentence[idx],
					edit_distance
				)

		return corrupted, error_locations

	def evaluate_ranking(self, test_data: list[list[str]], corrupted_data: list[list[str]], error_locations: list[list[int]], unigram_weight: float, k_values: list[int] = [3, 5, 10]):
		total_errors = 0
		corrected_errors = 0
		covered_errors = 0
		sentence_hits = 0
		mrr_sum = 0

		top_k_hits = {k: 0 for k in k_values}

		for sentence, truth, indices in tqdm(zip(corrupted_data, test_data, error_locations), total=len(test_data), desc="Evaluating ranking"):
			sentence_ok = True
			lower_sentence = [word.lower() for word in sentence]

			for idx in indices:
				total_errors += 1
				true_word = truth[idx].lower()

				ranked = self.spell_corrector.get_ranked_candidates(
					sentence=sentence,
					position=idx,
					unigram_weight=unigram_weight,
					lower_sentence=lower_sentence
				)
				
				rank = None
				for i, (candidate, _) in enumerate(ranked, start=1):
					if candidate.lower() == true_word:
						rank = i
						break

				if rank == 1:
					corrected_errors += 1
				else:
					sentence_ok = False

				if rank is not None:
					covered_errors += 1
					mrr_sum += 1 / rank

					for k in k_values:
						if rank <= k:
							top_k_hits[k] += 1

			if sentence_ok:
				sentence_hits += 1

		metrics = {
			"accuracy@1": corrected_errors / total_errors,
			"sentence_accuracy": sentence_hits / len(test_data),
			"coverage": covered_errors / total_errors,
			"mrr": mrr_sum / total_errors,
			"ranking_accuracy": corrected_errors / covered_errors
		}

		for k in k_values:
			metrics[f"accuracy@{k}"] = top_k_hits[k] / total_errors

		return metrics
	

In [96]:
evaluator = SpellCorrectionEvaluator(
	spell_corrector=spell_corrector,
	seed=42
)


In [97]:
evaluator.random_typo("hola", 2)


'hóala'

In [98]:
corrupted, locations = evaluator.generate_errors(
	test_sentences,
	errors_per_sentence=1
)

print(corrupted[:3])


Generating errors:   1%|          | 668/59474 [00:00<00:09, 6445.55it/s]

Generating errors: 100%|██████████| 59474/59474 [00:08<00:00, 7015.77it/s]

[['para', 'quién', 'xes'], ['supongo', 'que', 'crees', 'que', 'cuando', 'pasaz', 'los', 'hombres', 'simplemente', 'caen', 'rendidos', 'a', 'tus', 'pies'], ['está', 'seguro', 'que', 'sería', 'capaz', 'de', 'usarlo', 'a', 'sangrí', 'fría']]


In [99]:
N = 1000

metrics = evaluator.evaluate_ranking(
    test_sentences if N is None else test_sentences[:N],
    corrupted if N is None else corrupted[:N],
    locations if N is None else locations[:N],
    unigram_weight=0.8
)


Evaluating ranking:   0%|          | 0/1000 [00:00<?, ?it/s]

Evaluating ranking: 100%|██████████| 1000/1000 [00:00<00:00, 2085.86it/s]


In [100]:
print(metrics)


{'accuracy@1': 0.887, 'sentence_accuracy': 0.887, 'coverage': 0.975, 'mrr': 0.9247428571428575, 'ranking_accuracy': 0.9097435897435897, 'accuracy@3': 0.965, 'accuracy@5': 0.972, 'accuracy@10': 0.975}
